In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC datetimes
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) metric.csv summaries
metric_unique_cmdb = metric_df['cmdb_id'].dropna().unique()
metric_unique_kpi = metric_df['kpi_name'].dropna().unique()
metric_head = metric_df.loc[:, ['timestamp','cmdb_id','kpi_name','value']].head(10)

# Compute global stats per (cmdb_id, kpi_name)
_metric_group = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(['count','mean','max', lambda x: x.quantile(0.95)]).reset_index()
_metric_group = _metric_group.rename(columns={'<lambda_0>':'P95'} if '<lambda_0>' in _metric_group.columns else {'<lambda>':'P95'})
# Ensure columns ordering
metric_group_stats = _metric_group[['cmdb_id','kpi_name','count','P95','mean','max']].sort_values(by='count', ascending=False).head(20)

# 2) trace.csv summaries
trace_unique_cmdb = trace_df['cmdb_id'].dropna().unique()
trace_unique_names = trace_df['trace_name'].dropna().unique()
trace_head = trace_df.loc[:, ['timestamp','cmdb_id','trace_name','value']].head(10)

_trace_group = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(['count','mean','max', lambda x: x.quantile(0.95)]).reset_index()
_trace_group = _trace_group.rename(columns={'<lambda_0>':'P95'} if '<lambda_0>' in _trace_group.columns else {'<lambda>':'P95'})
trace_group_stats = _trace_group[['cmdb_id','trace_name','count','P95','mean','max']].sort_values(by='count', ascending=False).head(20)

# 3) log.csv summaries
log_unique_cmdb = log_df['cmdb_id'].dropna().unique()
log_unique_names = log_df['log_name'].dropna().unique()
log_head = log_df.loc[:, ['timestamp','cmdb_id','log_name','value']].head(10)

_log_group = log_df.groupby(['cmdb_id','log_name'])['value'].agg(['count','mean','max', lambda x: x.quantile(0.95)]).reset_index()
_log_group = _log_group.rename(columns={'<lambda_0>':'P95'} if '<lambda_0>' in _log_group.columns else {'<lambda>':'P95'})
log_group_stats = _log_group[['cmdb_id','log_name','count','P95','mean','max']].sort_values(by='count', ascending=False).head(20)

# 4) error_logs.csv summaries
error_count = error_df.shape[0]
error_unique_cmdb = error_df['cmdb_id'].dropna().unique()
error_head = error_df.loc[:, ['timestamp','cmdb_id','message']].head(10)

# Display compact summaries (as variables)
metric_unique_cmdb, metric_unique_kpi, metric_head, metric_group_stats, \
trace_unique_cmdb, trace_unique_names, trace_head, trace_group_stats, \
log_unique_cmdb, log_unique_names, log_head, log_group_stats, \
error_count, error_unique_cmdb, error_head

```
Out[1]:
```
# Compose a concise plain-English summary based on the previously loaded telemetry variables
# Reuse existing DataFrames: metric_df, trace_df, log_df, error_df, metric_group_stats, trace_group_stats, log_group_stats

# Basic counts and name lists
metric_cmdb_count = metric_df['cmdb_id'].nunique()
metric_cmdb_list = list(metric_df['cmdb_id'].unique())
metric_kpi_list = sorted(metric_df['kpi_name'].unique())

trace_cmdb_count = trace_df['cmdb_id'].nunique()
trace_cmdb_list = list(trace_df['cmdb_id'].unique())
trace_name_list = sorted(trace_df['trace_name'].unique())

log_cmdb_count = log_df['cmdb_id'].nunique()
log_cmdb_list = list(log_df['cmdb_id'].unique())
log_name_list = sorted(log_df['log_name'].unique())

error_rows = error_df.shape[0]
error_cmdb_list = list(error_df['cmdb_id'].unique())

# Notable metric series by mean and max (top 3 each) for compact highlights
metric_stats = metric_df.groupby(['cmdb_id','kpi_name'])['value'].agg(['count','mean','max']).reset_index()
top3_by_mean = metric_stats.sort_values('mean', ascending=False).head(3)
top3_by_max = metric_stats.sort_values('max', ascending=False).head(3)

# Notable trace series (highlight large row_count series and latency durations)
trace_stats = trace_df.groupby(['cmdb_id','trace_name'])['value'].agg(['count','mean','max']).reset_index()
top_trace_rowcount = trace_stats[trace_stats['trace_name'].str.contains('row_count')].sort_values('mean', ascending=False).head(3)
top_trace_latency = trace_stats[trace_stats['trace_name'].str.contains('duration_mean|duration_p95')].sort_values('mean', ascending=False).head(3)

# Notable log series (top row_count means)
log_stats = log_df.groupby(['cmdb_id','log_name'])['value'].agg(['count','mean','max']).reset_index()
top_log_rowcount = log_stats[log_stats['log_name'].str.contains('row_count')].sort_values('mean', ascending=False).head(3)

# Build summary string (compact)
summary = (
    f"Summary of loaded telemetry (UTC timestamps):\n\n"
    f"1) metric.csv\n"
    f"- Unique cmdb_id: {metric_cmdb_count} (sample: {metric_cmdb_list[:10]}{' +%d more' % (len(metric_cmdb_list)-10) if len(metric_cmdb_list)>10 else ''}).\n"
    f"- Unique kpi_name ({len(metric_kpi_list)}): {metric_kpi_list}.\n"
    f"- Per-series sample counts: most metric series have count = 25 (seen in aggregated stats).\n"
    f"- Notable metric series by mean (top 3):\n"
    f"  {', '.join([f'{r.cmdb_id}/{r.kpi_name} mean={r.mean:.3g} max={r.max:.3g} count={int(r['count'])}' for _, r in top3_by_mean.iterrows()])}.\n"
    f"- Notable metric series by max (top 3):\n"
    f"  {', '.join([f'{r.cmdb_id}/{r.kpi_name} max={r.max:.3g} mean={r.mean:.3g} count={int(r['count'])}' for _, r in top3_by_max.iterrows()])}.\n\n"
    f"2) trace.csv\n"
    f"- Unique cmdb_id: {trace_cmdb_count} (sample: {trace_cmdb_list[:10]}{' +%d more' % (len(trace_cmdb_list)-10) if len(trace_cmdb_list)>10 else ''}).\n"
    f"- Unique trace_name: {len(trace_name_list)} names (sample first 10: {trace_name_list[:10]}).\n"
    f"- Many trace error_rate series show 0; notable row_count series (top by mean):\n"
    f"  {', '.join([f'{r.cmdb_id}/{r.trace_name} mean={r.mean:.3g} max={r.max:.3g} count={int(r['count'])}' for _, r in top_trace_rowcount.iterrows()])}.\n"
    f"- Notable trace latency-type series by mean (top 3):\n"
    f"  {', '.join([f'{r.cmdb_id}/{r.trace_name} mean={r.mean:.3g} max={r.max:.3g}' for _, r in top_trace_latency.iterrows()])}.\n\n"
    f"3) log.csv\n"
    f"- Unique cmdb_id: {log_cmdb_count} (sample: {log_cmdb_list[:10]}{' +%d more' % (len(log_cmdb_list)-10) if len(log_cmdb_list)>10 else ''}).\n"
    f"- Unique log_name: {log_name_list}.\n"
    f"- log.error_count is 0 for observed series; log.row_count shows high traffic for several services (top examples):\n"
    f"  {', '.join([f'{r.cmdb_id}/{r.log_name} mean={r.mean:.3g} max={r.max:.3g} count={int(r['count'])}' for _, r in top_log_rowcount.iterrows()])}.\n\n"
    f"4) error_logs.csv\n"
    f"- Total rows: {error_rows}. Unique cmdb_id in error logs: {error_cmdb_list}.\n"
    f"- Current dataset: error_logs.csv contains 0 rows (no raw error messages present).\n\n"
    f"Overall quick takeaways:\n"
    f"- The telemetry contains many services (~{metric_cmdb_count} metric-side services) with 8 KPI types (cpu, mem, diskio, socket, workload, latency-50, latency-90, error).\n"
    f"- Most metric/trace/log series have 25 samples in the window provided. Many trace error_rate and log.error_count series are zero.\n"
    f"- High activity examples: ts-travel2-service shows very high CPU mean/max and high socket values and large trace row_count; several services (ts-basic-service, ts-travel-service, ts-travel2-service, ts-auth-service) have large log.row_count means indicating heavy request traffic.\n"
    f"- No error log rows are present to inspect further.\n\n"
    f"If you want, I can now (a) show the exact top-20 aggregated metric table, (b) filter by a time window, or (c) dig into specific services noted above."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(array(['ts-admin-basic-info-service', 'ts-admin-order-service',
       'ts-admin-route-service', 'ts-admin-travel-service',
       'ts-admin-user-service', 'ts-assurance-mongo',
       'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service',
       'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service',
       'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo',
       'ts-consign-price-mongo', 'ts-consign-price-service',
       'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service',
       'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service',
       'ts-food-mongo', 'ts-food-mysql', 'ts-food-service',
       'ts-inside-payment-mongo', 'ts-inside-payment-service',
       'ts-news-service', 'ts-notification-mongo',
       'ts-notification-service', 'ts-order-mongo',
       'ts-order-other-mongo', 'ts-order-other-service',
       'ts-order-service', 'ts-payment-mongo', 'ts-payment-service',
       'ts-preserve-mongo', 'ts-preserve-other-mongo',
       'ts-preserve-other-service', 'ts-preserve-service',
       'ts-price-mongo', 'ts-price-service', 'ts-rebook-service',
       'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service',
       'ts-seat-service', 'ts-security-mongo', 'ts-security-service',
       'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo',
       'ts-ticket-office-service', 'ts-ticketinfo-service',
       'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo',
       'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo',
       'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo',
       'ts-user-service', 'ts-verification-code-service',
       'ts-voucher-mysql', 'ts-voucher-service'], dtype=object), array(['cpu', 'diskio', 'error', 'latency-50', 'latency-90', 'mem',
       'socket', 'workload'], dtype=object),                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service         cpu  7.045773e-01
1 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service      diskio  7.025277e+04
2 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service       error  0.000000e+00
3 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service  latency-50  1.729082e-02
4 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service  latency-90  2.345816e-02
5 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service         mem  2.358354e+08
6 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service      socket  8.000000e+00
7 2024-01-23 17:58:00+00:00  ts-admin-basic-info-service    workload  7.904286e-01
8 2024-01-23 17:58:00+00:00       ts-admin-order-service         cpu  9.507774e-02
9 2024-01-23 17:58:00+00:00       ts-admin-order-service      diskio  5.399292e+04,                          cmdb_id    kpi_name  count           P95          mean           max
359           ts-voucher-service      socket     25  4.000000e+00  4.000000e+00  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  7.258677e-01  4.856667e-01  7.639249e-01
1    ts-admin-basic-info-service      diskio     25  6.343846e+04  5.160158e+04  7.025277e+04
2    ts-admin-basic-info-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25  1.780597e-02  1.723667e-02  1.790469e-02
4    ts-admin-basic-info-service  latency-90     25  2.405074e-02  2.349796e-02  2.422845e-02
5    ts-admin-basic-info-service         mem     25  2.390244e+08  2.376777e+08  2.390424e+08
6    ts-admin-basic-info-service      socket     25  7.933333e+00  6.865481e+00  8.150000e+00
320             ts-travel2-mongo         mem     25  6.744983e+07  6.728291e+07  6.749143e+07
321             ts-travel2-mongo      socket     25  8.000000e+00  8.000000e+00  8.000000e+00
322           ts-travel2-service         cpu     25  5.045677e+00  4.417904e+00  5.232377e+00
323           ts-travel2-service      diskio     25  5.915491e+04  5.022833e+04  6.008266e+04
324           ts-travel2-service  latency-50     25  1.775848e-02  1.695209e-02  1.779727e-02
325           ts-travel2-service  latency-90     25  3.724715e-01  3.697351e-01  3.739009e-01
326           ts-travel2-service         mem     25  2.709551e+08  2.705129e+08  2.712015e+08
327           ts-travel2-service      socket     25  1.936000e+01  1.793221e+01  1.955000e+01
312            ts-travel-service       error     25  0.000000e+00  0.000000e+00  0.000000e+00
313            ts-travel-service  latency-50     25  1.747013e-02  1.634807e-02  1.752399e-02
314            ts-travel-service  latency-90     25  2.633361e-01  1.548209e-01  2.691644e-01
315            ts-travel-service         mem     25  2.727408e+08  2.724443e+08  2.727447e+08, array(['root', 'ts-admin-travel-service', 'ts-auth-service',
       'ts-basic-service', 'ts-config-service',
       'ts-consign-price-service', 'ts-consign-service',
       'ts-food-map-service', 'ts-food-service',
       'ts-inside-payment-service', 'ts-order-other-service',
       'ts-order-service', 'ts-price-service', 'ts-route-service',
       'ts-seat-service', 'ts-station-service', 'ts-ticketinfo-service',
       'ts-train-service', 'ts-travel-service', 'ts-travel2-service',
       'ts-admin-basic-info-service', 'ts-assurance-service',
       'ts-contacts-service', 'ts-payment-service',
       'ts-preserve-other-service', 'ts-preserve-service',
       'ts-security-service', 'ts-user-service'], dtype=object), array(['trace.to_ts-admin-travel-service.duration_mean',
       'trace.to_ts-admin-travel-service.duration_p95',
       'trace.to_ts-admin-travel-service.error_rate',
       'trace.to_ts-admin-travel-service.row_count',
       'trace.to_ts-auth-service.duration_mean',
       'trace.to_ts-auth-service.duration_p95',
       'trace.to_ts-auth-service.error_rate',
       'trace.to_ts-auth-service.row_count',
       'trace.to_ts-consign-service.duration_mean',
       'trace.to_ts-consign-service.duration_p95',
       'trace.to_ts-consign-service.error_rate',
       'trace.to_ts-consign-service.row_count',
       'trace.to_ts-food-service.duration_mean',
       'trace.to_ts-food-service.duration_p95',
       'trace.to_ts-food-service.error_rate',
       'trace.to_ts-food-service.row_count',
       'trace.to_ts-inside-payment-service.duration_mean',
       'trace.to_ts-inside-payment-service.duration_p95',
       'trace.to_ts-inside-payment-service.error_rate',
       'trace.to_ts-inside-payment-service.row_count',
       'trace.to_ts-order-service.duration_mean',
       'trace.to_ts-order-service.duration_p95',
       'trace.to_ts-order-service.error_rate',
       'trace.to_ts-order-service.row_count',
       'trace.to_ts-ticketinfo-service.duration_mean',
       'trace.to_ts-ticketinfo-service.duration_p95',
       'trace.to_ts-ticketinfo-service.error_rate',
       'trace.to_ts-ticketinfo-service.row_count',
       'trace.to_ts-travel-service.duration_mean',
       'trace.to_ts-travel-service.duration_p95',
       'trace.to_ts-travel-service.error_rate',
       'trace.to_ts-travel-service.row_count',
       'trace.to_ts-travel2-service.duration_mean',
       'trace.to_ts-travel2-service.duration_p95',
       'trace.to_ts-travel2-service.error_rate',
       'trace.to_ts-travel2-service.row_count',
       'trace.from_root.duration_mean', 'trace.from_root.duration_p95',
       'trace.from_root.error_rate', 'trace.from_root.row_count',
       'trace.from_ts-admin-travel-service.duration_mean',
       'trace.from_ts-admin-travel-service.duration_p95',
       'trace.from_ts-admin-travel-service.error_rate',
       'trace.from_ts-admin-travel-service.row_count',
       'trace.from_ts-auth-service.duration_mean',
       'trace.from_ts-auth-service.duration_p95',
       'trace.from_ts-auth-service.error_rate',
       'trace.from_ts-auth-service.row_count',
       'trace.from_ts-basic-service.duration_mean',
       'trace.from_ts-basic-service.duration_p95',
       'trace.from_ts-basic-service.error_rate',
       'trace.from_ts-basic-service.row_count',
       'trace.from_ts-ticketinfo-service.duration_mean',
       'trace.from_ts-ticketinfo-service.duration_p95',
       'trace.from_ts-ticketinfo-service.error_rate',
       'trace.from_ts-ticketinfo-service.row_count',
       'trace.to_ts-basic-service.duration_mean',
       'trace.to_ts-basic-service.duration_p95',
       'trace.to_ts-basic-service.error_rate',
       'trace.to_ts-basic-service.row_count',
       'trace.to_ts-price-service.duration_mean',
       'trace.to_ts-price-service.duration_p95',
       'trace.to_ts-price-service.error_rate',
       'trace.to_ts-price-service.row_count',
       'trace.to_ts-route-service.duration_mean',
       'trace.to_ts-route-service.duration_p95',
       'trace.to_ts-route-service.error_rate',
       'trace.to_ts-route-service.row_count',
       'trace.to_ts-station-service.duration_mean',
       'trace.to_ts-station-service.duration_p95',
       'trace.to_ts-station-service.error_rate',
       'trace.to_ts-station-service.row_count',
       'trace.to_ts-train-service.duration_mean',
       'trace.to_ts-train-service.duration_p95',
       'trace.to_ts-train-service.error_rate',
       'trace.to_ts-train-service.row_count',
       'trace.from_ts-config-service.duration_mean',
       'trace.from_ts-config-service.duration_p95',
       'trace.from_ts-config-service.error_rate',
       'trace.from_ts-config-service.row_count',
       'trace.from_ts-seat-service.duration_mean',
       'trace.from_ts-seat-service.duration_p95',
       'trace.from_ts-seat-service.error_rate',
       'trace.from_ts-seat-service.row_count',
       'trace.to_ts-config-service.duration_mean',
       'trace.to_ts-config-service.duration_p95',
       'trace.to_ts-config-service.error_rate',
       'trace.to_ts-config-service.row_count',
       'trace.from_ts-consign-price-service.duration_mean',
       'trace.from_ts-consign-price-service.duration_p95',
       'trace.from_ts-consign-price-service.error_rate',
       'trace.from_ts-consign-price-service.row_count',
       'trace.from_ts-consign-service.duration_mean',
       'trace.from_ts-consign-service.duration_p95',
       'trace.from_ts-consign-service.error_rate',
       'trace.from_ts-consign-service.row_count',
       'trace.to_ts-consign-price-service.duration_mean',
       'trace.to_ts-consign-price-service.duration_p95',
       'trace.to_ts-consign-price-service.error_rate',
       'trace.to_ts-consign-price-service.row_count',
       'trace.from_ts-food-map-service.duration_mean',
       'trace.from_ts-food-map-service.duration_p95',
       'trace.from_ts-food-map-service.error_rate',
       'trace.from_ts-food-map-service.row_count',
       'trace.from_ts-food-service.duration_mean',
       'trace.from_ts-food-service.duration_p95',
       'trace.from_ts-food-service.error_rate',
       'trace.from_ts-food-service.row_count',
       'trace.to_ts-food-map-service.duration_mean',
       'trace.to_ts-food-map-service.duration_p95',
       'trace.to_ts-food-map-service.error_rate',
       'trace.to_ts-food-map-service.row_count',
       'trace.from_ts-inside-payment-service.duration_mean',
       'trace.from_ts-inside-payment-service.duration_p95',
       'trace.from_ts-inside-payment-service.error_rate',
       'trace.from_ts-inside-payment-service.row_count',
       'trace.from_ts-order-other-service.duration_mean',
       'trace.from_ts-order-other-service.duration_p95',
       'trace.from_ts-order-other-service.error_rate',
       'trace.from_ts-order-other-service.row_count',
       'trace.from_ts-travel2-service.duration_mean',
       'trace.from_ts-travel2-service.duration_p95',
       'trace.from_ts-travel2-service.error_rate',
       'trace.from_ts-travel2-service.row_count',
       'trace.to_ts-order-other-service.duration_mean',
       'trace.to_ts-order-other-service.duration_p95',
       'trace.to_ts-order-other-service.error_rate',
       'trace.to_ts-order-other-service.row_count',
       'trace.from_ts-order-service.duration_mean',
       'trace.from_ts-order-service.duration_p95',
       'trace.from_ts-order-service.error_rate',
       'trace.from_ts-order-service.row_count',
       'trace.from_ts-travel-service.duration_mean',
       'trace.from_ts-travel-service.duration_p95',
       'trace.from_ts-travel-service.error_rate',
       'trace.from_ts-travel-service.row_count',
       'trace.from_ts-price-service.duration_mean',
       'trace.from_ts-price-service.duration_p95',
       'trace.from_ts-price-service.error_rate',
       'trace.from_ts-price-service.row_count',
       'trace.from_ts-route-service.duration_mean',
       'trace.from_ts-route-service.duration_p95',
       'trace.from_ts-route-service.error_rate',
       'trace.from_ts-route-service.row_count',
       'trace.to_ts-seat-service.duration_mean',
       'trace.to_ts-seat-service.duration_p95',
       'trace.to_ts-seat-service.error_rate',
       'trace.to_ts-seat-service.row_count',
       'trace.from_ts-station-service.duration_mean',
       'trace.from_ts-station-service.duration_p95',
       'trace.from_ts-station-service.error_rate',
       'trace.from_ts-station-service.row_count',
       'trace.from_ts-train-service.duration_mean',
       'trace.from_ts-train-service.duration_p95',
       'trace.from_ts-train-service.error_rate',
       'trace.from_ts-train-service.row_count',
       'trace.to_ts-admin-basic-info-service.duration_mean',
       'trace.to_ts-admin-basic-info-service.duration_p95',
       'trace.to_ts-admin-basic-info-service.error_rate',
       'trace.to_ts-admin-basic-info-service.row_count',
       'trace.to_ts-assurance-service.duration_mean',
       'trace.to_ts-assurance-service.duration_p95',
       'trace.to_ts-assurance-service.error_rate',
       'trace.to_ts-assurance-service.row_count',
       'trace.to_ts-contacts-service.duration_mean',
       'trace.to_ts-contacts-service.duration_p95',
       'trace.to_ts-contacts-service.error_rate',
       'trace.to_ts-contacts-service.row_count',
       'trace.to_ts-preserve-other-service.duration_mean',
       'trace.to_ts-preserve-other-service.duration_p95',
       'trace.to_ts-preserve-other-service.error_rate',
       'trace.to_ts-preserve-other-service.row_count',
       'trace.to_ts-preserve-service.duration_mean',
       'trace.to_ts-preserve-service.duration_p95',
       'trace.to_ts-preserve-service.error_rate',
       'trace.to_ts-preserve-service.row_count',
       'trace.from_ts-admin-basic-info-service.duration_mean',
       'trace.from_ts-admin-basic-info-service.duration_p95',
       'trace.from_ts-admin-basic-info-service.error_rate',
       'trace.from_ts-admin-basic-info-service.row_count',
       'trace.from_ts-assurance-service.duration_mean',
       'trace.from_ts-assurance-service.duration_p95',
       'trace.from_ts-assurance-service.error_rate',
       'trace.from_ts-assurance-service.row_count',
       'trace.from_ts-preserve-other-service.duration_mean',
       'trace.from_ts-preserve-other-service.duration_p95',
       'trace.from_ts-preserve-other-service.error_rate',
       'trace.from_ts-preserve-other-service.row_count',
       'trace.from_ts-preserve-service.duration_mean',
       'trace.from_ts-preserve-service.duration_p95',
       'trace.from_ts-preserve-service.error_rate',
       'trace.from_ts-preserve-service.row_count',
       'trace.from_ts-contacts-service.duration_mean',
       'trace.from_ts-contacts-service.duration_p95',
       'trace.from_ts-contacts-service.error_rate',
       'trace.from_ts-contacts-service.row_count',
       'trace.to_ts-payment-service.duration_mean',
       'trace.to_ts-payment-service.duration_p95',
       'trace.to_ts-payment-service.error_rate',
       'trace.to_ts-payment-service.row_count',
       'trace.from_ts-security-service.duration_mean',
       'trace.from_ts-security-service.duration_p95',
       'trace.from_ts-security-service.error_rate',
       'trace.from_ts-security-service.row_count',
       'trace.from_ts-payment-service.duration_mean',
       'trace.from_ts-payment-service.duration_p95',
       'trace.from_ts-payment-service.error_rate',
       'trace.from_ts-payment-service.row_count',
       'trace.to_ts-security-service.duration_mean',
       'trace.to_ts-security-service.duration_p95',
       'trace.to_ts-security-service.error_rate',
       'trace.to_ts-security-service.row_count',
       'trace.to_ts-user-service.duration_mean',
       'trace.to_ts-user-service.duration_p95',
       'trace.to_ts-user-service.error_rate',
       'trace.to_ts-user-service.row_count',
       'trace.from_ts-user-service.duration_mean',
       'trace.from_ts-user-service.duration_p95',
       'trace.from_ts-user-service.error_rate',
       'trace.from_ts-user-service.row_count'], dtype=object),                   timestamp cmdb_id                                      trace_name      value
0 2024-01-23 17:58:00+00:00    root  trace.to_ts-admin-travel-service.duration_mean   0.148141
1 2024-01-23 17:58:00+00:00    root   trace.to_ts-admin-travel-service.duration_p95   0.148141
2 2024-01-23 17:58:00+00:00    root     trace.to_ts-admin-travel-service.error_rate   0.000000
3 2024-01-23 17:58:00+00:00    root      trace.to_ts-admin-travel-service.row_count   1.000000
4 2024-01-23 17:58:00+00:00    root          trace.to_ts-auth-service.duration_mean   0.426747
5 2024-01-23 17:58:00+00:00    root           trace.to_ts-auth-service.duration_p95   0.745390
6 2024-01-23 17:58:00+00:00    root             trace.to_ts-auth-service.error_rate   0.000000
7 2024-01-23 17:58:00+00:00    root              trace.to_ts-auth-service.row_count  13.000000
8 2024-01-23 17:58:00+00:00    root       trace.to_ts-consign-service.duration_mean   0.022993
9 2024-01-23 17:58:00+00:00    root        trace.to_ts-consign-service.duration_p95   0.023334,                 cmdb_id                                        trace_name  count          P95         mean          max
728  ts-travel2-service          trace.from_ts-seat-service.duration_mean     25     0.010669     0.010033     0.011640
729  ts-travel2-service           trace.from_ts-seat-service.duration_p95     25     0.015181     0.013241     0.015853
730  ts-travel2-service             trace.from_ts-seat-service.error_rate     25     0.000000     0.000000     0.000000
731  ts-travel2-service              trace.from_ts-seat-service.row_count     25   354.800000   305.640000   356.000000
732  ts-travel2-service       trace.from_ts-travel2-service.duration_mean     25     0.010242     0.009847     0.010292
733  ts-travel2-service        trace.from_ts-travel2-service.duration_p95     25     0.047457     0.046018     0.047873
734  ts-travel2-service          trace.from_ts-travel2-service.error_rate     25     0.000000     0.000000     0.000000
735  ts-travel2-service           trace.from_ts-travel2-service.row_count     25  2860.400000  2484.960000  2879.000000
720  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25     0.067862     0.064824     0.084397
721  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25     0.080110     0.076231     0.160705
722  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25     0.000000     0.000000     0.000000
723  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25    11.000000     9.120000    11.000000
712   ts-travel-service          trace.to_ts-travel-service.duration_mean     25     0.010317     0.009722     0.011057
713   ts-travel-service           trace.to_ts-travel-service.duration_p95     25     0.048028     0.046497     0.049868
714   ts-travel-service             trace.to_ts-travel-service.error_rate     25     0.000000     0.000000     0.000000
715   ts-travel-service              trace.to_ts-travel-service.row_count     25  1713.000000  1346.800000  2090.000000
716  ts-travel2-service                     trace.from_root.duration_mean     25     0.307404     0.294678     0.307904
717  ts-travel2-service                      trace.from_root.duration_p95     25     0.376388     0.337520     0.433641
718  ts-travel2-service                        trace.from_root.error_rate     25     0.000000     0.000000     0.000000
719  ts-travel2-service                         trace.from_root.row_count     25    80.000000    71.800000    83.000000, array(['ts-admin-travel-service', 'ts-assurance-mongo', 'ts-auth-service',
       'ts-basic-service', 'ts-consign-mongo', 'ts-consign-price-mongo',
       'ts-consign-service', 'ts-food-map-mongo', 'ts-food-map-service',
       'ts-food-mongo', 'ts-food-service', 'ts-inside-payment-mongo',
       'ts-inside-payment-service', 'ts-notification-mongo',
       'ts-order-other-mongo', 'ts-order-other-service',
       'ts-order-service', 'ts-payment-mongo', 'ts-preserve-other-mongo',
       'ts-price-mongo', 'ts-price-service', 'ts-route-service',
       'ts-seat-service', 'ts-security-mongo', 'ts-ticket-office-mongo',
       'ts-train-mongo', 'ts-travel-service', 'ts-travel2-mongo',
       'ts-travel2-service', 'ts-ui-dashboard',
       'ts-admin-basic-info-service', 'ts-assurance-service',
       'ts-auth-mongo', 'ts-config-mongo', 'ts-contacts-mongo',
       'ts-contacts-service', 'ts-order-mongo', 'ts-preserve-mongo',
       'ts-preserve-other-service', 'ts-preserve-service',
       'ts-route-mongo', 'ts-security-service', 'ts-station-mongo',
       'ts-travel-mongo', 'ts-user-mongo', 'ts-consign-price-service'],
      dtype=object), array(['log.error_count', 'log.row_count'], dtype=object),                   timestamp                  cmdb_id         log_name  value
0 2024-01-23 17:58:00+00:00  ts-admin-travel-service  log.error_count      0
1 2024-01-23 17:58:00+00:00  ts-admin-travel-service    log.row_count      3
2 2024-01-23 17:58:00+00:00       ts-assurance-mongo  log.error_count      0
3 2024-01-23 17:58:00+00:00       ts-assurance-mongo    log.row_count      1
4 2024-01-23 17:58:00+00:00          ts-auth-service  log.error_count      0
5 2024-01-23 17:58:00+00:00          ts-auth-service    log.row_count     44
6 2024-01-23 17:58:00+00:00         ts-basic-service  log.error_count      0
7 2024-01-23 17:58:00+00:00         ts-basic-service    log.row_count    227
8 2024-01-23 17:58:00+00:00         ts-consign-mongo  log.error_count      0
9 2024-01-23 17:58:00+00:00         ts-consign-mongo    log.row_count      1,                     cmdb_id         log_name  count     P95     mean   max
3   ts-admin-travel-service    log.row_count     25    33.0    27.36    33
2   ts-admin-travel-service  log.error_count     25     0.0     0.00     0
11          ts-auth-service    log.row_count     25   356.0   311.68   358
10          ts-auth-service  log.error_count     25     0.0     0.00     0
23       ts-consign-service    log.row_count     25    63.0    42.00    77
22       ts-consign-service  log.error_count     25     0.0     0.00     0
13         ts-basic-service    log.row_count     25  2698.8  2313.72  2942
12         ts-basic-service  log.error_count     25     0.0     0.00     0
87       ts-travel2-service    log.row_count     25  2493.6  2178.12  2524
86       ts-travel2-service  log.error_count     25     0.0     0.00     0
83        ts-travel-service    log.row_count     25  1937.2  1522.20  2314
82        ts-travel-service  log.error_count     25     0.0     0.00     0
63         ts-price-service    log.row_count     25   298.8   253.20   326
62         ts-price-service  log.error_count     25     0.0     0.00     0
66         ts-route-service  log.error_count     25     0.0     0.00     0
67         ts-route-service    log.row_count     25  1080.4   951.56  1155
69          ts-seat-service    log.row_count     25  1993.8  1720.68  2244
68          ts-seat-service  log.error_count     25     0.0     0.00     0
49         ts-order-service    log.row_count     25   414.8   335.08   536
48         ts-order-service  log.error_count     25     0.0     0.00     0, 0, array([], dtype=object), Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Incident window (UTC)
start = pd.Timestamp("2024-01-23 17:55:53", tz="UTC")
end = pd.Timestamp("2024-01-23 18:25:53", tz="UTC")

# 1) Metric anomalies
metric_win = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
# Merge P95 from full-series stats
metric_merge = metric_win.merge(metric_group_stats[['cmdb_id','kpi_name','P95']], on=['cmdb_id','kpi_name'], how='left')
metric_merge['is_anom'] = metric_merge['value'] >= metric_merge['P95']

metric_group_win = metric_merge.groupby(['cmdb_id','kpi_name']).agg(
    count_in_window=('value','size'),
    anomaly_count=('is_anom','sum'),
    max_in_window=('value','max'),
    P95=('P95','first')
).reset_index()

# earliest anomaly timestamp
earliest_metric = metric_merge[metric_merge['is_anom']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})

metric_anoms_df = metric_group_win.merge(earliest_metric, on=['cmdb_id','kpi_name'], how='left')
metric_anoms_df = metric_anoms_df[metric_anoms_df['anomaly_count']>0].sort_values(by='anomaly_count', ascending=False).head(20)
# Keep columns order compact
metric_anoms_df = metric_anoms_df[['cmdb_id','kpi_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_in_window','P95']]

# 2) Trace anomalies
trace_win = trace_df[(trace_df['timestamp'] >= start) & (trace_df['timestamp'] <= end)].copy()
trace_merge = trace_win.merge(trace_group_stats[['cmdb_id','trace_name','P95']], on=['cmdb_id','trace_name'], how='left')

# Conditions per trace_name type
cond_duration = trace_merge['trace_name'].str.contains('duration_mean|duration_p95', regex=True, na=False) & (trace_merge['value'] >= trace_merge['P95'])
cond_error = trace_merge['trace_name'].str.contains('error_rate', na=False) & (trace_merge['value'] > 0)
cond_rowcount = trace_merge['trace_name'].str.contains('row_count', na=False) & (trace_merge['value'] >= trace_merge['P95'])
trace_merge['is_anom'] = cond_duration | cond_error | cond_rowcount

trace_group_win = trace_merge.groupby(['cmdb_id','trace_name']).agg(
    count_in_window=('value','size'),
    anomaly_count=('is_anom','sum'),
    max_in_window=('value','max'),
    P95=('P95','first')
).reset_index()

earliest_trace = trace_merge[trace_merge['is_anom']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})

trace_anoms_df = trace_group_win.merge(earliest_trace, on=['cmdb_id','trace_name'], how='left')
trace_anoms_df = trace_anoms_df[trace_anoms_df['anomaly_count']>0].sort_values(by='anomaly_count', ascending=False).head(20)
trace_anoms_df = trace_anoms_df[['cmdb_id','trace_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_in_window','P95']]

# 3) Log anomalies
# Compute global P5 per (cmdb_id,log_name) using full series (rule 9)
log_p5 = log_df.groupby(['cmdb_id','log_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'P5'})
log_thresholds = log_group_stats[['cmdb_id','log_name','P95']].merge(log_p5, on=['cmdb_id','log_name'], how='left')

log_win = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()
log_merge = log_win.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

cond_rowcount_lr = log_merge['log_name'].str.contains('row_count', na=False) & ((log_merge['value'] <= log_merge['P5']) | (log_merge['value'] >= log_merge['P95']))
cond_errorcount = log_merge['log_name'].str.contains('error_count', na=False) & (log_merge['value'] > 0)
log_merge['is_anom'] = cond_rowcount_lr | cond_errorcount

log_group_win = log_merge.groupby(['cmdb_id','log_name']).agg(
    count_in_window=('value','size'),
    anomaly_count=('is_anom','sum'),
    earliest_anom_ts=('timestamp', lambda x: x[log_merge.loc[x.index,'is_anom']].min() if any(log_merge.loc[x.index,'is_anom']) else pd.NaT),
    min_in_window=('value','min'),
    max_in_window=('value','max'),
    P95=('P95','first'),
    P5=('P5','first')
).reset_index()

# For earliest we computed via agg above; ensure column name consistent
# If lambda approach above produced issues, compute earliest separately
earliest_log = log_merge[log_merge['is_anom']].groupby(['cmdb_id','log_name'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
# Merge with group stats
log_group_win = log_group_win.merge(earliest_log, on=['cmdb_id','log_name'], how='left')

# Use earliest_anomaly_timestamp column consistently
if 'earliest_anomaly_timestamp' not in log_group_win.columns:
    log_group_win = log_group_win.rename(columns={'earliest_anom_ts':'earliest_anomaly_timestamp'})

log_anoms_df = log_group_win[['cmdb_id','log_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','min_in_window','max_in_window','P95','P5']]
log_anoms_df = log_anoms_df[log_anoms_df['anomaly_count']>0].sort_values(by='anomaly_count', ascending=False).head(20)

# 4) Candidate summary: distinct cmdb_id appearing in any anomaly outputs
metric_counts = metric_anoms_df.groupby('cmdb_id').size().reset_index(name='metric_anomaly_series_count')
trace_counts = trace_anoms_df.groupby('cmdb_id').size().reset_index(name='trace_anomaly_series_count')
log_counts = log_anoms_df.groupby('cmdb_id').size().reset_index(name='log_anomaly_series_count')

candidates = pd.DataFrame({'cmdb_id': pd.unique(
    pd.concat([metric_anoms_df['cmdb_id'], trace_anoms_df['cmdb_id'], log_anoms_df['cmdb_id']]).values
)})

candidate_summary_df = candidates.merge(metric_counts, on='cmdb_id', how='left').merge(trace_counts, on='cmdb_id', how='left').merge(log_counts, on='cmdb_id', how='left')
candidate_summary_df = candidate_summary_df.fillna(0)
# ensure integer counts
for col in ['metric_anomaly_series_count','trace_anomaly_series_count','log_anomaly_series_count']:
    candidate_summary_df[col] = candidate_summary_df[col].astype(int)

candidate_summary_df = candidate_summary_df.sort_values(by=['metric_anomaly_series_count','trace_anomaly_series_count','log_anomaly_series_count'], ascending=False).head(10)

# Display compact results
metric_anoms_df, trace_anoms_df, log_anoms_df, candidate_summary_df

```
Out[2]:
```
Summary of anomaly detection (window 2024-01-23 17:55:53 → 2024-01-23 18:25:53 UTC)

What was done
- Used full-series P95 thresholds (computed from entire files) and inspected points inside the incident window.
- Flagged metric points >= full-series P95; trace anomalies include duration >= P95, error_rate > 0, row_count >= P95; log anomalies include row_count <= P5 or >= P95 and error_count > 0.
- Returned only series with anomalies and aggregated counts / earliest timestamps / max/min values.

Key findings — metric anomalies
- Several services show metric series exceeding their full-series P95 inside the window.
- Notable metric results:
  - ts-admin-basic-info-service: 7 metric series flagged (cpu, diskio, mem, socket, latency-50/90, error). Earliest anomalies around 2024-01-23T17:58:00Z / 18:00–18:20 for different KPIs. Example: cpu P95=0.7259, max in-window 0.7639.
  - ts-travel2-service: 6 metric series flagged (cpu, diskio, mem, socket, latency-50/90). Example: socket P95≈19.36, max in-window 19.55 (earliest anomaly 18:01:00Z).
  - ts-travel-service and ts-travel2-mongo also show multiple metric anomalies.
  - ts-voucher-service.socket: anomaly_count = 25 (P95=4.0, max=4.0) — constant at the threshold.
- Caution: some "error" metric series have full-series P95 = 0.0, so all zero-valued points are counted as anomalies (e.g., ts-admin-basic-info-service/error, ts-travel-service/error). These should be interpreted carefully (P95 == 0 makes every 0 >= P95).

Key findings — trace anomalies
- ts-travel2-service dominates trace anomalies:
  - trace.from_root.row_count: count_in_window=25, anomaly_count=5, earliest anomaly 2024-01-23T18:00:00Z, max=83, P95=80.
  - trace.from_ts-admin-travel-service.row_count: anomaly_count=3 (earliest 18:02:00Z).
  - Several duration_mean / duration_p95 series for ts-travel2-service and ts-travel-service exceed their P95s (examples: from_root.duration_mean, from_root.duration_p95, to_ts-travel-service.duration_p95). Example: ts-travel-service trace.to_ts-travel-service.duration_p95 P95=0.04803, max in-window=0.04987 (earliest 17:59:00Z).

Key findings — log anomalies
- Many services show log.row_count anomalies (either unusually high or low relative to full-series P5/P95):
  - ts-consign-service: anomaly_count=6 (earliest 17:58:00Z), log.row_count P95=63.0, P5=15.4, min=14, max=77.
  - ts-admin-travel-service: anomaly_count=5, log.row_count P95=33.0, P5=22.4.
  - ts-auth-service: anomaly_count=5, log.row_count P95=356.0, P5=277.6.
  - ts-basic-service, ts-order-service, ts-price-service, ts-route-service, ts-seat-service, ts-travel-service, ts-travel2-service also show multiple log.row_count anomalies (high traffic counts).
- log.error_count series were 0 in the dataset (no error_count anomalies flagged).

Top candidate services to investigate (compact counts per service)
- ts-admin-basic-info-service: metric_anomaly_series_count = 7, trace = 0, log = 0
- ts-travel2-service: metric = 6, trace = 12, log = 1
- ts-travel-service: metric = 4, trace = 3, log = 1
- ts-travel2-mongo: metric = 2, trace = 0, log = 0
- ts-voucher-service: metric = 1, trace = 0, log = 0
- ts-consign-service: metric = 0, trace = 0, log = 1
- ts-admin-travel-service: metric = 0, trace = 0, log = 1
- ts-auth-service: metric = 0, trace = 0, log = 1
- ts-basic-service: metric = 0, trace = 0, log = 1
- ts-order-service: metric = 0, trace = 0, log = 1

Short interpretation and recommended next steps
- The strongest signals point to the travel-related services, especially ts-travel2-service and ts-travel-service, which have the most trace anomalies (high row_count and elevated durations) and multiple metric anomalies (CPU, memory, diskio, socket, latencies). These are the highest-priority suspects.
- ts-admin-basic-info-service shows many metric anomalies (including cpu, mem, diskio, socket and latency). Investigate its resource usage and whether its P95 baselines are appropriate.
- Several services show large log.row_count deviations (ts-auth, ts-basic, ts-travel, ts-travel2, ts-order, etc.) indicating heavy traffic or spikes that may relate to service load.
- Re-check "error" metric anomalies where full-series P95 == 0 (those are likely artefacts of the thresholding rule). Treat zero-P95 series with caution before acting on them.
- Suggested follow-ups: inspect traces for ts-travel2-service flows around 18:00–18:05 UTC, review resource metrics (cpu/mem/socket/diskio) for ts-travel2 and ts-travel at the earliest anomaly timestamps, and correlate high log.row_count services with request spikes and downstream latency/error behavior.

If you want, I can:
- show the exact top-20 anomaly rows (tables) for metric/trace/log with the numeric columns,
- expand the investigation to surrounding minutes, or
- filter anomalies excluding series with P95 == 0 to avoid the zero-P95 artefacts.

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_in_window           P95
2    ts-admin-basic-info-service       error               25             25  2024-01-23 17:58:00+00:00   0.000000e+00  0.000000e+00
359           ts-voucher-service      socket               25             25  2024-01-23 17:58:00+00:00   4.000000e+00  4.000000e+00
321             ts-travel2-mongo      socket               25             25  2024-01-23 17:58:00+00:00   8.000000e+00  8.000000e+00
312            ts-travel-service       error               25             25  2024-01-23 17:58:00+00:00   0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50               25              2  2024-01-23 17:59:00+00:00   1.790469e-02  1.780597e-02
1    ts-admin-basic-info-service      diskio               25              2  2024-01-23 17:58:00+00:00   7.025277e+04  6.343846e+04
0    ts-admin-basic-info-service         cpu               25              2  2024-01-23 18:00:00+00:00   7.639249e-01  7.258677e-01
4    ts-admin-basic-info-service  latency-90               25              2  2024-01-23 17:59:00+00:00   2.422845e-02  2.405074e-02
313            ts-travel-service  latency-50               25              2  2024-01-23 18:11:00+00:00   1.752399e-02  1.747013e-02
314            ts-travel-service  latency-90               25              2  2024-01-23 18:03:00+00:00   2.691644e-01  2.633361e-01
5    ts-admin-basic-info-service         mem               25              2  2024-01-23 18:19:00+00:00   2.390424e+08  2.390244e+08
6    ts-admin-basic-info-service      socket               25              2  2024-01-23 17:58:00+00:00   8.150000e+00  7.933333e+00
320             ts-travel2-mongo         mem               25              2  2024-01-23 18:20:00+00:00   6.749143e+07  6.744983e+07
315            ts-travel-service         mem               25              2  2024-01-23 18:19:00+00:00   2.727447e+08  2.727408e+08
323           ts-travel2-service      diskio               25              2  2024-01-23 18:00:00+00:00   6.008266e+04  5.915491e+04
322           ts-travel2-service         cpu               25              2  2024-01-23 18:04:00+00:00   5.232377e+00  5.045677e+00
324           ts-travel2-service  latency-50               25              2  2024-01-23 18:05:00+00:00   1.779727e-02  1.775848e-02
325           ts-travel2-service  latency-90               25              2  2024-01-23 18:05:00+00:00   3.739009e-01  3.724715e-01
326           ts-travel2-service         mem               25              2  2024-01-23 18:19:00+00:00   2.712015e+08  2.709551e+08
327           ts-travel2-service      socket               25              2  2024-01-23 18:01:00+00:00   1.955000e+01  1.936000e+01,                 cmdb_id                                        trace_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_in_window          P95
719  ts-travel2-service                         trace.from_root.row_count               25              5  2024-01-23 18:00:00+00:00      83.000000    80.000000
723  ts-travel2-service      trace.from_ts-admin-travel-service.row_count               25              3  2024-01-23 18:02:00+00:00      11.000000    11.000000
713   ts-travel-service           trace.to_ts-travel-service.duration_p95               25              2  2024-01-23 17:59:00+00:00       0.049868     0.048028
715   ts-travel-service              trace.to_ts-travel-service.row_count               25              2  2024-01-23 17:59:00+00:00    2090.000000  1713.000000
716  ts-travel2-service                     trace.from_root.duration_mean               25              2  2024-01-23 18:04:00+00:00       0.307904     0.307404
717  ts-travel2-service                      trace.from_root.duration_p95               25              2  2024-01-23 17:59:00+00:00       0.433641     0.376388
712   ts-travel-service          trace.to_ts-travel-service.duration_mean               25              2  2024-01-23 17:59:00+00:00       0.011057     0.010317
720  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean               25              2  2024-01-23 18:04:00+00:00       0.084397     0.067862
721  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95               25              2  2024-01-23 18:00:00+00:00       0.160705     0.080110
728  ts-travel2-service          trace.from_ts-seat-service.duration_mean               25              2  2024-01-23 17:59:00+00:00       0.011640     0.010669
729  ts-travel2-service           trace.from_ts-seat-service.duration_p95               25              2  2024-01-23 18:05:00+00:00       0.015853     0.015181
731  ts-travel2-service              trace.from_ts-seat-service.row_count               25              2  2024-01-23 18:01:00+00:00     356.000000   354.800000
732  ts-travel2-service       trace.from_ts-travel2-service.duration_mean               25              2  2024-01-23 18:04:00+00:00       0.010292     0.010242
733  ts-travel2-service        trace.from_ts-travel2-service.duration_p95               25              2  2024-01-23 18:08:00+00:00       0.047873     0.047457
735  ts-travel2-service           trace.from_ts-travel2-service.row_count               25              2  2024-01-23 18:01:00+00:00    2879.000000  2860.400000,                     cmdb_id       log_name  count_in_window  anomaly_count earliest_anomaly_timestamp  min_in_window  max_in_window     P95      P5
23       ts-consign-service  log.row_count               25              6  2024-01-23 17:58:00+00:00             14             77    63.0    15.4
3   ts-admin-travel-service  log.row_count               25              5  2024-01-23 17:58:00+00:00              3             33    33.0    22.4
11          ts-auth-service  log.row_count               25              5  2024-01-23 17:58:00+00:00             44            358   356.0   277.6
13         ts-basic-service  log.row_count               25              4  2024-01-23 17:58:00+00:00            227           2942  2698.8  2021.8
49         ts-order-service  log.row_count               25              4  2024-01-23 17:58:00+00:00             39            536   414.8   263.8
63         ts-price-service  log.row_count               25              4  2024-01-23 17:58:00+00:00             24            326   298.8   218.8
67         ts-route-service  log.row_count               25              4  2024-01-23 17:58:00+00:00            103           1155  1080.4   834.6
69          ts-seat-service  log.row_count               25              4  2024-01-23 17:58:00+00:00            175           2244  1993.8  1497.6
83        ts-travel-service  log.row_count               25              4  2024-01-23 17:58:00+00:00            160           2314  1937.2  1207.4
87       ts-travel2-service  log.row_count               25              4  2024-01-23 17:58:00+00:00            234           2524  2493.6  1911.0,                        cmdb_id  metric_anomaly_series_count  trace_anomaly_series_count  log_anomaly_series_count
0  ts-admin-basic-info-service                            7                           0                         0
4           ts-travel2-service                            6                          12                         1
3            ts-travel-service                            4                           3                         1
2             ts-travel2-mongo                            2                           0                         0
1           ts-voucher-service                            1                           0                         0
5           ts-consign-service                            0                           0                         1
6      ts-admin-travel-service                            0                           0                         1
7              ts-auth-service                            0                           0                         1
8             ts-basic-service                            0                           0                         1
9             ts-order-service                            0                           0                         1)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Incident window (UTC)
start = pd.Timestamp("2024-01-23 17:55:53", tz="UTC")
end = pd.Timestamp("2024-01-23 18:25:53", tz="UTC")

services = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Ensure metric_group_stats has P95 column (it was created earlier as metric_group_stats)
# We'll use metric_group_stats[['cmdb_id','kpi_name','P95']]
thresholds = metric_group_stats[['cmdb_id','kpi_name','P95']].copy()

# Filter metric_df to incident window once
metric_win_all = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

all_runs = []
per_kpi_summaries = []

service_summaries = []

for svc in services:
    svc_rows = metric_df[metric_df['cmdb_id'] == svc]
    if svc_rows.shape[0] == 0:
        # Explicit note for services with no metric rows
        service_summaries.append({
            'cmdb_id': svc,
            'note': 'no_metric_rows',
            'number_of_faulty_kpis': 0,
            'earliest_fault_start': pd.NaT,
            'worst_severity': np.nan
        })
        continue

    # KPIs available for this service (from thresholds)
    svc_kpis = thresholds[thresholds['cmdb_id'] == svc]['kpi_name'].unique().tolist()
    svc_faulty_kpis = 0
    svc_earliest = None
    svc_worst_sev = -np.inf

    for kpi in svc_kpis:
        # global P95
        p95_row = thresholds[(thresholds['cmdb_id'] == svc) & (thresholds['kpi_name'] == kpi)]
        if p95_row.empty:
            P95 = np.nan
        else:
            P95 = float(p95_row['P95'].iloc[0])

        # rows in window for this series
        df_s = metric_win_all[(metric_win_all['cmdb_id'] == svc) & (metric_win_all['kpi_name'] == kpi)].copy().sort_values('timestamp')
        count_in_window = df_s.shape[0]
        if count_in_window == 0:
            anomaly_count = 0
            earliest_anom_ts = pd.NaT
            max_in_window = np.nan
            # still record series-level summary
            per_kpi_summaries.append({
                'cmdb_id': svc,
                'kpi_name': kpi,
                'count_in_window': int(count_in_window),
                'anomaly_count': int(anomaly_count),
                'earliest_anomaly_timestamp': earliest_anom_ts,
                'max_in_window': max_in_window,
                'P95': P95
            })
            continue

        # Mark anomalies (value >= P95). If P95 is nan, mark False.
        if pd.isna(P95):
            df_s['is_anom'] = False
        else:
            df_s['is_anom'] = df_s['value'] >= P95

        anomaly_count = int(df_s['is_anom'].sum())
        earliest_anom_ts = df_s.loc[df_s['is_anom'], 'timestamp'].min() if anomaly_count>0 else pd.NaT
        max_in_window = df_s['value'].max()

        per_kpi_summaries.append({
            'cmdb_id': svc,
            'kpi_name': kpi,
            'count_in_window': int(count_in_window),
            'anomaly_count': int(anomaly_count),
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'max_in_window': float(max_in_window),
            'P95': P95
        })

        # Identify consecutive anomaly runs only if there is at least one anomaly
        if anomaly_count > 0:
            df_an = df_s[df_s['is_anom']].copy().reset_index(drop=True)
            # compute gap in seconds between consecutive anomaly timestamps
            df_an['dt_s'] = df_an['timestamp'].diff().dt.total_seconds().fillna(0)
            # consider consecutive if dt_s == 60 (one minute)
            df_an['new_run'] = (df_an['dt_s'] != 60).astype(int)
            df_an['run_id'] = df_an['new_run'].cumsum()

            runs = df_an.groupby('run_id').agg(
                run_start_timestamp=('timestamp','first'),
                run_end_timestamp=('timestamp','last'),
                run_length=('timestamp','size'),
                run_max=('value','max')
            ).reset_index(drop=True)

            # compute severity
            def compute_sev(row):
                if pd.isna(P95):
                    return np.nan
                if P95 == 0:
                    return 'P95_zero'
                return float((row['run_max'] - P95) / P95)

            runs['run_severity'] = runs.apply(compute_sev, axis=1)
            runs['cmdb_id'] = svc
            runs['kpi_name'] = kpi
            # Append runs to all_runs
            for _, r in runs.iterrows():
                all_runs.append({
                    'cmdb_id': r['cmdb_id'],
                    'kpi_name': r['kpi_name'],
                    'run_start_timestamp': r['run_start_timestamp'],
                    'run_end_timestamp': r['run_end_timestamp'],
                    'run_length': int(r['run_length']),
                    'run_max': float(r['run_max']),
                    'run_severity': r['run_severity'],
                    'P95': P95,
                    'anomaly_count': anomaly_count,
                    'count_in_window': count_in_window
                })

            # Update service-level aggregates if KPI meets condition:
            # Only consider KPIs where run_length >=2 OR anomaly_count >=2
            meets_condition = (runs['run_length'].max() >= 2) or (anomaly_count >= 2)
            if meets_condition:
                svc_faulty_kpis += 1
                # earliest run start among this KPI
                kpi_earliest = runs['run_start_timestamp'].min()
                if pd.isna(svc_earliest) or svc_earliest is None:
                    svc_earliest = kpi_earliest
                else:
                    if pd.notna(kpi_earliest) and kpi_earliest < svc_earliest:
                        svc_earliest = kpi_earliest
                # worst severity: consider numeric severities; treat 'P95_zero' as inf
                # determine max severity among runs
                sev_values = []
                for sv in runs['run_severity'].values:
                    if sv == 'P95_zero':
                        sev_values.append(np.inf)
                    elif pd.isna(sv):
                        continue
                    else:
                        sev_values.append(float(sv))
                if sev_values:
                    max_sev = max(sev_values)
                    if np.isinf(max_sev):
                        svc_worst_sev = np.inf
                    else:
                        svc_worst_sev = max(svc_worst_sev, max_sev)

    # finalize service summary
    service_summaries.append({
        'cmdb_id': svc,
        'note': '' if svc_rows.shape[0]>0 else 'no_metric_rows',
        'number_of_faulty_kpis': int(svc_faulty_kpis),
        'earliest_fault_start': svc_earliest if svc_earliest is not None else pd.NaT,
        'worst_severity': (np.inf if np.isinf(svc_worst_sev) else (svc_worst_sev if svc_worst_sev>-np.inf else np.nan))
    })

# Build DataFrames
runs_df = pd.DataFrame(all_runs)
# Filter to only KPIs meeting the condition: run_length >=2 OR anomaly_count >=2
if not runs_df.empty:
    runs_df = runs_df[(runs_df['run_length'] >= 2) | (runs_df['anomaly_count'] >= 2)].copy()
# sort and limit (prefer up to 20)
if not runs_df.empty:
    # sort by run_length desc, then run_severity (treat 'P95_zero' as highest)
    def sev_sort_key(val):
        if val == 'P95_zero':
            return np.inf
        try:
            return float(val)
        except:
            return -np.inf
    runs_df['sev_sort'] = runs_df['run_severity'].apply(sev_sort_key)
    runs_df = runs_df.sort_values(by=['run_length','sev_sort'], ascending=[False, False]).drop(columns=['sev_sort']).reset_index(drop=True)
    runs_df_selected = runs_df[['cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length','run_max','run_severity','P95','anomaly_count','count_in_window']].head(20)
else:
    runs_df_selected = runs_df[['cmdb_id','kpi_name']].head(0)

per_kpi_df = pd.DataFrame(per_kpi_summaries)
# Only keep KPIs where run_length >=2 OR anomaly_count >=2 for compact return
if not runs_df_selected.empty:
    # identify KPIs included
    included = runs_df_selected[['cmdb_id','kpi_name']].drop_duplicates()
    per_kpi_df = per_kpi_df.merge(included, on=['cmdb_id','kpi_name'], how='inner')
else:
    # fallback: include KPIs with anomaly_count >=2
    per_kpi_df = per_kpi_df[per_kpi_df['anomaly_count'] >= 2].copy().reset_index(drop=True)

service_summary_df = pd.DataFrame(service_summaries)
# Compact formatting: convert timestamps to UTC ISO (they are already timezone-aware)
# Keep selected columns
service_summary_df = service_summary_df[['cmdb_id','note','number_of_faulty_kpis','earliest_fault_start','worst_severity']]

# Display compact outputs
runs_df_selected, per_kpi_df[['cmdb_id','kpi_name','count_in_window','anomaly_count','earliest_anomaly_timestamp','max_in_window','P95']].head(50), service_summary_df

```
Out[3]:
```
Focused fault-identification results for the five services (window 2024-01-23 17:55:53 → 2024-01-23 18:25:53 UTC)

High-level answer
- Four services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service) showed no metric KPIs meeting the fault criteria in this incident window (no faulty KPIs found in the available metrics).
- ts-travel-service is the only service with metric anomalies detected in the window. It has 4 KPIs flagged for investigation: error, latency-50, latency-90, and mem.

Details for ts-travel-service (compact)
- Faulty KPIs count: 4
- Earliest fault start: 2024-01-23T17:58:00Z
- Worst severity: infinite (artifact due to P95 == 0 for the error metric; see note below)

KPIs and runs (selected compact details)
- error
  - Full-series P95 = 0.0
  - count_in_window = 25, anomaly_count = 25
  - One long consecutive run: run_start = 2024-01-23T17:58:00Z, run_end = 2024-01-23T18:22:00Z, run_length = 25
  - run_max = 0.0, run_severity = P95_zero (P95 == 0 → severity is a special case / artifact)
  - Interpretation: this is a thresholding artifact because P95 is zero; every zero point meets value >= P95.

- latency-90
  - Full-series P95 ≈ 0.2633361
  - count_in_window = 25, anomaly_count = 2
  - Two isolated anomaly points (not consecutive): earliest anomaly 2024-01-23T18:03:00Z, max_in_window ≈ 0.2691644
  - These are short spikes (run_length = 1 each) but there are multiple spike points (anomaly_count ≥ 2).

- latency-50
  - Full-series P95 ≈ 0.01747013
  - count_in_window = 25, anomaly_count = 2
  - Two isolated anomaly points, earliest 2024-01-23T18:11:00Z, max_in_window ≈ 0.01752399
  - Short spikes (not sustained).

- mem
  - Full-series P95 ≈ 272,740,800 (2.727408e+08)
  - count_in_window = 25, anomaly_count = 2
  - Two isolated high values, earliest anomaly 2024-01-23T18:19:00Z, max_in_window ≈ 272,744,700
  - Very small relative exceedance (severity near zero).

Notes & interpretation
- The largest “severity” signal is the error metric P95 == 0 which makes every 0 value flagged (artifact). Treat P95 == 0 series cautiously — they typically do not indicate a real positive spike.
- Latency KPIs (latency-90 and latency-50) show a couple of isolated spikes (two points each) around 18:03–18:15 UTC — this suggests short-lived latency incidents rather than sustained degradation.
- mem shows two tiny exceedances around 18:19 UTC (very small relative increase over P95).
- The other four services have no metric anomalies in this window per the applied rule (value >= full-series P95 and run/anomaly thresholds).

Recommended next steps
1. Exclude or treat P95 == 0 series (like the error KPI) as a special case to avoid false-positives. Confirm whether error KPI baseline is meaningful.
2. For ts-travel-service:
   - Investigate latency spikes (latency-90 around 18:03 UTC and latency-50 around 18:11–18:15 UTC). Correlate with traces and logs for ts-travel-service and downstream services around those timestamps.
   - Check memory around 18:19 UTC for potential transient pressure.
3. If you want, I can: (a) show the exact numeric rows for the runs and KPI aggregates, (b) exclude P95==0 artifacts and re-run, or (c) correlate these timestamps with trace/log anomalies already observed (recommended next step).

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id    kpi_name       run_start_timestamp         run_end_timestamp  run_length       run_max run_severity           P95  anomaly_count  count_in_window
0  ts-travel-service       error 2024-01-23 17:58:00+00:00 2024-01-23 18:22:00+00:00          25  0.000000e+00     P95_zero  0.000000e+00             25               25
1  ts-travel-service  latency-90 2024-01-23 18:03:00+00:00 2024-01-23 18:03:00+00:00           1  2.691644e-01     0.022133  2.633361e-01              2               25
2  ts-travel-service  latency-90 2024-01-23 18:05:00+00:00 2024-01-23 18:05:00+00:00           1  2.641855e-01     0.003225  2.633361e-01              2               25
3  ts-travel-service  latency-50 2024-01-23 18:11:00+00:00 2024-01-23 18:11:00+00:00           1  1.752399e-02     0.003083  1.747013e-02              2               25
4  ts-travel-service  latency-50 2024-01-23 18:15:00+00:00 2024-01-23 18:15:00+00:00           1  1.751809e-02     0.002745  1.747013e-02              2               25
5  ts-travel-service         mem 2024-01-23 18:22:00+00:00 2024-01-23 18:22:00+00:00           1  2.727447e+08     0.000014  2.727408e+08              2               25
6  ts-travel-service         mem 2024-01-23 18:19:00+00:00 2024-01-23 18:19:00+00:00           1  2.727422e+08     0.000005  2.727408e+08              2               25,              cmdb_id    kpi_name  count_in_window  anomaly_count earliest_anomaly_timestamp  max_in_window           P95
0  ts-travel-service       error               25             25  2024-01-23 17:58:00+00:00   0.000000e+00  0.000000e+00
1  ts-travel-service  latency-50               25              2  2024-01-23 18:11:00+00:00   1.752399e-02  1.747013e-02
2  ts-travel-service  latency-90               25              2  2024-01-23 18:03:00+00:00   2.691644e-01  2.633361e-01
3  ts-travel-service         mem               25              2  2024-01-23 18:19:00+00:00   2.727447e+08  2.727408e+08,              cmdb_id note  number_of_faulty_kpis      earliest_fault_start  worst_severity
0    ts-auth-service                           0                       NaT             inf
1   ts-order-service                           0                       NaT             inf
2   ts-route-service                           0                       NaT             inf
3   ts-train-service                           0                       NaT             inf
4  ts-travel-service                           4 2024-01-23 17:58:00+00:00             inf)```
```